# MiniMax-H3 FL2VA LoRA Training (DiffSynth-Studio)

Trains a MiniMax-H3-NF4 FL2VA LoRA using the current upstream DiffSynth-Studio
recipe. Verified against DiffSynth-Studio commit `b8e3811e5c4abd83a44b99b2bcff7ccabdb26a71`
— see `docs/diffsynth_h3_api_notes.md` in the repo for the full API mapping and
any deltas from assumptions in Ticket 002.

**Do not full-finetune H3.** This notebook is LoRA-only.

Responsibilities: setup -> dataset validation -> preprocessing/cache -> LoRA
training -> checkpoint validation generation. For everyday generation
(no training), use `H3_Inference.ipynb` instead.

## A — Runtime

In [ ]:
import subprocess, sys, shutil

print("=== nvidia-smi ===")
try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30).stdout)
except Exception as e:
    print("nvidia-smi failed:", e)

print("=== torch / cuda ===")
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("device:", props.name, "| total VRAM GB:", round(props.total_memory / (1024**3), 2))
else:
    print("WARNING: no GPU. MiniMax-H3 training/inference requires a GPU runtime.")
    print("In Colab: Runtime > Change runtime type > GPU, then re-run this cell.")

print("=== host RAM ===")
try:
    with open("/proc/meminfo") as f:
        for line in f:
            if line.startswith(("MemTotal", "MemAvailable")):
                print(line.strip())
except Exception as e:
    print("meminfo failed:", e)

print("=== disk ===")
total, used, free = shutil.disk_usage("/")
print("total GB:", round(total/(1024**3), 2), "free GB:", round(free/(1024**3), 2))
print("=== python ===")
print(sys.version)


## B — Google Drive (persistent storage)

Mounts Drive and defines explicit persistent paths for datasets, model cache,
training cache, experiments, and checkpoints. Nothing here deletes existing
Drive content.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT = "/content/drive/MyDrive/minimax-h3"
DATASETS_DIR = os.path.join(DRIVE_ROOT, "datasets")
MODELS_CACHE_DIR = os.path.join(DRIVE_ROOT, "models")
TRAINING_CACHE_DIR = os.path.join(DRIVE_ROOT, "training-cache")
EXPERIMENTS_DIR = os.path.join(DRIVE_ROOT, "experiments")
CHECKPOINTS_DIR = os.path.join(DRIVE_ROOT, "checkpoints")

for path in [DATASETS_DIR, MODELS_CACHE_DIR, TRAINING_CACHE_DIR, EXPERIMENTS_DIR, CHECKPOINTS_DIR]:
    os.makedirs(path, exist_ok=True)
    print("ok:", path)

# Route Hugging Face / ModelScope caches through Drive so large downloads
# survive runtime restarts instead of re-downloading every session.
os.environ.setdefault("MODELSCOPE_CACHE", os.path.join(MODELS_CACHE_DIR, "modelscope_cache"))
os.environ.setdefault("HF_HOME", os.path.join(MODELS_CACHE_DIR, "huggingface_cache"))

# DiffSynth-Studio's own ModelConfig.download_if_necessary() caches to a
# cwd-relative "./models/<model_id>/..." path by default -- NOT governed by
# MODELSCOPE_CACHE/HF_HOME above. A training subprocess (cwd=DiffSynth-Studio)
# and this notebook kernel (cwd=/content) would otherwise each download their
# own ~32GB copy. DIFFSYNTH_MODEL_BASE_PATH overrides this to a single
# absolute, Drive-backed path that both respect regardless of cwd, and that
# survives runtime restarts. Verified against commit b8e3811
# (diffsynth/core/loader/config.py: reset_local_model_path).
os.environ.setdefault("DIFFSYNTH_MODEL_BASE_PATH", os.path.join(MODELS_CACHE_DIR, "diffsynth_models"))


## C — DiffSynth-Studio

Clones (or updates) the current official repository and installs it. Records
the exact commit SHA — do not assume upstream training arguments are
unchanged from what's documented in `docs/diffsynth_h3_api_notes.md`.

In [ ]:
import subprocess

REPO_DIR = "/content/DiffSynth-Studio"

def run(cmd, cwd=None, check=True):
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    print(result.stdout[-3000:])
    if result.returncode != 0:
        print("STDERR:", result.stderr[-3000:])
        if check:
            raise RuntimeError(f"command failed: {' '.join(cmd)}")
    return result

if not os.path.isdir(REPO_DIR):
    run(["git", "clone", "https://github.com/modelscope/DiffSynth-Studio.git", REPO_DIR])
else:
    run(["git", "pull"], cwd=REPO_DIR)

commit_sha = run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR).stdout.strip()
print("DiffSynth-Studio commit:", commit_sha)

run(["pip", "install", "-e", ".[all]", "--quiet"], cwd=REPO_DIR)

# peft raises ImportError during LoRA injection if an outdated torchao is
# importable at all (regardless of whether this run uses torchao
# quantization). Verified against peft 0.19.1 requiring torchao>=0.16.0.
run(["pip", "install", "-U", "torchao>=0.16.0", "--quiet"], check=False)

import importlib
diffsynth_version = importlib.metadata.version("diffsynth")
print("diffsynth package version:", diffsynth_version)


## D — Dataset

Select the dataset directory and validate it before doing anything else. The
check below mirrors `scripts/validate_dataset.py` in the project repo
(kept inline here so this notebook is self-contained and doesn't depend on
the repo being synced into this Colab session). If you change
`scripts/validate_dataset.py`, update this cell to match.

Expected layout (verified against the current DiffSynth-Studio H3 FL2VA
example dataset):

```
<DATASET_DIR>/
    metadata.csv   # columns: video, prompt, [input_audio], [frame_rate]
    video_0001.mp4
    ...
```

`input_image` / `end_image` are **not** metadata columns — DiffSynth derives
first/last-frame conditioning automatically from each training video.

**Training does not proceed past this cell if the dataset is invalid.**

In [ ]:
# @markdown Dataset folder name under `MyDrive/minimax-h3/datasets/`
DATASET_NAME = "my-motion-lora-v1"  # @param {type:"string"}
TARGET_NUM_FRAMES = 124  # @param {type:"integer"}

DATASET_DIR = os.path.join(DATASETS_DIR, DATASET_NAME)
print("Dataset dir:", DATASET_DIR)


In [ ]:
import csv, json as _json, shutil as _shutil, statistics, subprocess as _subprocess

H3_VALID_FRAME_COUNTS = sorted(17 * n + 5 for n in range(1, 10))
if TARGET_NUM_FRAMES not in H3_VALID_FRAME_COUNTS:
    raise ValueError(
        f"TARGET_NUM_FRAMES={TARGET_NUM_FRAMES} is not a valid H3 frame count (17n+5). "
        f"Nearest valid values: {H3_VALID_FRAME_COUNTS}"
    )

FFPROBE_AVAILABLE = _shutil.which("ffprobe") is not None

def _probe_video(video_path):
    cmd = ["ffprobe", "-v", "error", "-print_format", "json", "-show_format", "-show_streams", video_path]
    result = _subprocess.run(cmd, capture_output=True, text=True, timeout=30)
    if result.returncode != 0:
        return {"error": result.stderr.strip()[:300]}
    data = _json.loads(result.stdout)
    vstreams = [s for s in data.get("streams", []) if s.get("codec_type") == "video"]
    astreams = [s for s in data.get("streams", []) if s.get("codec_type") == "audio"]
    if not vstreams:
        return {"error": "no video stream"}
    v = vstreams[0]
    duration = float(data.get("format", {}).get("duration", 0) or 0)
    fps = None
    rate = v.get("avg_frame_rate") or v.get("r_frame_rate")
    if rate and rate != "0/0":
        num, _, den = rate.partition("/")
        fps = float(num) / float(den) if den and float(den) else float(num)
    nb_frames = v.get("nb_frames")
    frame_count = int(nb_frames) if nb_frames and nb_frames.isdigit() else (round(duration * fps) if fps else None)
    return {
        "width": v.get("width"), "height": v.get("height"),
        "duration_sec": round(duration, 3), "fps": round(fps, 3) if fps else None,
        "frame_count": frame_count, "has_audio": len(astreams) > 0,
    }

def validate_dataset(dataset_dir, target_num_frames):
    metadata_path = os.path.join(dataset_dir, "metadata.csv")
    if not os.path.isdir(dataset_dir):
        raise FileNotFoundError(f"Dataset directory not found: {dataset_dir}")
    if not os.path.exists(metadata_path):
        raise FileNotFoundError(f"metadata.csv not found in {dataset_dir}")

    with open(metadata_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        columns = set(reader.fieldnames or [])

    missing_required = {"video", "prompt"} - columns
    if missing_required:
        raise ValueError(f"metadata.csv missing required columns: {sorted(missing_required)}")

    report = {
        "dataset_dir": dataset_dir, "ffprobe_available": FFPROBE_AVAILABLE,
        "video_count": len(rows), "valid_video_count": 0, "invalid_video_count": 0,
        "caption_coverage": 0.0, "audio_present_count": 0, "audio_missing_count": 0,
        "frame_count_compatible_count": 0, "entries": [],
    }
    caption_ok = 0
    for row in rows:
        problems = []
        video_rel = (row.get("video") or "").strip()
        prompt = (row.get("prompt") or "").strip()
        input_audio_rel = (row.get("input_audio") or "").strip()
        video_path = os.path.join(dataset_dir, video_rel) if video_rel else None
        if not video_rel:
            problems.append("empty video field")
        elif not os.path.exists(video_path):
            problems.append(f"video file does not exist: {video_path}")
        if not prompt:
            problems.append("missing/empty caption")
        if not input_audio_rel:
            # Verified against DiffSynth-Studio commit b8e3811: a blank
            # input_audio cell is read as NaN by pandas and crashes
            # ToAbsolutePath's os.path.join() before training starts.
            # LoadAudioWithTorchaudio already fails gracefully to None for a
            # real file with no audio track, so always point input_audio at
            # a real path (typically the video itself) instead.
            problems.append(
                "empty input_audio cell -- set it to the video's own filename "
                "(even if silent) rather than leaving it blank; combined with "
                "--silent_on_missing_audio the loader falls back to silence per-file"
            )
        media = None
        if not problems and FFPROBE_AVAILABLE:
            media = _probe_video(video_path)
            if "error" in media:
                problems.append(f"media probe failed: {media['error']}")
            else:
                if media["has_audio"]:
                    report["audio_present_count"] += 1
                else:
                    report["audio_missing_count"] += 1
                if media["frame_count"] is not None and media["frame_count"] < target_num_frames:
                    problems.append(f"only {media['frame_count']} frames, target is {target_num_frames}")
                else:
                    report["frame_count_compatible_count"] += 1
        if prompt:
            caption_ok += 1
        if problems:
            report["invalid_video_count"] += 1
        else:
            report["valid_video_count"] += 1
        report["entries"].append({"video": video_rel, "problems": problems, "media": media})

    report["caption_coverage"] = round(caption_ok / len(rows), 4) if rows else 0.0
    return report

dataset_report = validate_dataset(DATASET_DIR, TARGET_NUM_FRAMES)
print(f"videos: {dataset_report['video_count']}  valid: {dataset_report['valid_video_count']}  invalid: {dataset_report['invalid_video_count']}")
print(f"caption coverage: {dataset_report['caption_coverage']*100:.1f}%")
print(f"frame-count compatible: {dataset_report['frame_count_compatible_count']}")

if dataset_report["invalid_video_count"] > 0:
    print("\nInvalid entries:")
    for e in dataset_report["entries"]:
        if e["problems"]:
            print(f"  [FAIL] {e['video']}: {e['problems']}")
    raise RuntimeError(
        "Dataset has invalid entries. Fix the issues above before training. "
        "Training will NOT proceed with an invalid dataset."
    )

print("\nDataset OK. Safe to proceed.")


## E — Training config

Safe defaults from `CLAUDE.md`. Edit as needed for this experiment.

In [ ]:
EXPERIMENT_NAME = "h3-motion-lora-v1"  # @param {type:"string"}
CHECKPOINT_VARIANT = "NF4"  # @param ["NF4", "BF16"]
HEIGHT = 480  # @param {type:"integer"}
WIDTH = 832  # @param {type:"integer"}
NUM_FRAMES = 124  # @param {type:"integer"}
FPS = 24  # @param {type:"integer"}
LORA_RANK = 32  # @param {type:"integer"}
LEARNING_RATE = 1e-4  # @param {type:"number"}
NUM_EPOCHS = 5  # @param {type:"integer"}
DATASET_REPEAT = 100  # @param {type:"integer"}
SAVE_STEPS = 250  # @param {type:"integer"}
USE_GRADIENT_CHECKPOINTING = True  # @param {type:"boolean"}
USE_GRADIENT_CHECKPOINTING_OFFLOAD = False  # @param {type:"boolean"}
ENABLE_MODEL_CPU_OFFLOAD = False  # @param {type:"boolean"}
SILENT_ON_MISSING_AUDIO = True  # @param {type:"boolean"}
USE_TWO_STAGE_CACHE = False  # @param {type:"boolean"}
VALIDATION_INTERVAL_STEPS = 250  # @param {type:"integer"}
LORA_BASE_MODEL = "dit"
LORA_TARGET_MODULES = "qkv_proj,out_proj"

if NUM_FRAMES not in H3_VALID_FRAME_COUNTS:
    raise ValueError(f"NUM_FRAMES={NUM_FRAMES} invalid. Must be one of {H3_VALID_FRAME_COUNTS}")
if NUM_FRAMES != TARGET_NUM_FRAMES:
    print(f"NOTE: NUM_FRAMES ({NUM_FRAMES}) differs from the dataset validation target ({TARGET_NUM_FRAMES}).")

EXPERIMENT_DIR = os.path.join(EXPERIMENTS_DIR, EXPERIMENT_NAME)
CHECKPOINT_OUTPUT_DIR = os.path.join(CHECKPOINTS_DIR, EXPERIMENT_NAME)
CACHE_OUTPUT_DIR = os.path.join(TRAINING_CACHE_DIR, EXPERIMENT_NAME)
LOG_DIR = os.path.join(EXPERIMENT_DIR, "logs")
VALIDATION_OUTPUT_DIR = os.path.join(EXPERIMENT_DIR, "validation")

for path in [EXPERIMENT_DIR, CHECKPOINT_OUTPUT_DIR, LOG_DIR, VALIDATION_OUTPUT_DIR]:
    os.makedirs(path, exist_ok=True)

MODEL_ID_WITH_ORIGIN_PATHS = ",".join([
    "DiffSynth-Studio/MiniMax-H3-NF4:minimax-h3-text-encoder-nf4.safetensors",
    "DiffSynth-Studio/MiniMax-H3-NF4:minimax-h3-fl2va-nf4.safetensors",
    "DiffSynth-Studio/MiniMax-H3-NF4:video_vae_nf4.safetensors",
    "DiffSynth-Studio/MiniMax-H3-NF4:audio_vae_nf4.safetensors",
]) if CHECKPOINT_VARIANT == "NF4" else ",".join([
    "MiniMax/MiniMax-H3:FL2VA/text_encoder/model*.safetensors",
    "MiniMax/MiniMax-H3:FL2VA/video_vae/source/model.safetensors",
    "MiniMax/MiniMax-H3:FL2VA/audio_vae/model.safetensors",
    "MiniMax/MiniMax-H3:FL2VA/transformer/model*.safetensors",
])

config_record = dict(
    experiment_name=EXPERIMENT_NAME, checkpoint_variant=CHECKPOINT_VARIANT,
    height=HEIGHT, width=WIDTH, num_frames=NUM_FRAMES, fps=FPS,
    lora_base_model=LORA_BASE_MODEL, lora_target_modules=LORA_TARGET_MODULES,
    lora_rank=LORA_RANK, learning_rate=LEARNING_RATE, num_epochs=NUM_EPOCHS,
    dataset_repeat=DATASET_REPEAT, save_steps=SAVE_STEPS,
    use_gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
    use_gradient_checkpointing_offload=USE_GRADIENT_CHECKPOINTING_OFFLOAD,
    enable_model_cpu_offload=ENABLE_MODEL_CPU_OFFLOAD,
    silent_on_missing_audio=SILENT_ON_MISSING_AUDIO,
    use_two_stage_cache=USE_TWO_STAGE_CACHE,
    validation_interval_steps=VALIDATION_INTERVAL_STEPS,
    dataset_dir=DATASET_DIR, model_id_with_origin_paths=MODEL_ID_WITH_ORIGIN_PATHS,
    output_path=CHECKPOINT_OUTPUT_DIR, cache_dir=CACHE_OUTPUT_DIR,
)
with open(os.path.join(EXPERIMENT_DIR, "config.yaml"), "w") as f:
    import yaml
    yaml.safe_dump(config_record, f, sort_keys=False)
with open(os.path.join(EXPERIMENT_DIR, "dataset_summary.json"), "w") as f:
    _json.dump(dataset_report, f, indent=2)
print("Experiment config written to", EXPERIMENT_DIR)


## F — Preprocessing / Cache

The upstream **NF4** LoRA recipe (verified) runs single-stage — it loads the
NF4 model and processes each training batch on the fly, no separate cache
pass. The two-stage `data_process` -> `train` split is what the upstream
**BF16 full-precision** example uses, and `train.py` supports it generically
via `--task`. Enable `USE_TWO_STAGE_CACHE` above only if single-stage runs
out of memory (Phase 9 fallback ladder).

This cell only builds the command list; nothing executes here.

In [ ]:
def build_train_command(task, dataset_base_path, dataset_metadata_path, output_path, model_paths):
    cmd = [
        "accelerate", "launch", "examples/minimax_h3/model_training/train.py",
        "--dataset_base_path", dataset_base_path,
        "--data_file_keys", "video,input_audio",
        "--extra_inputs", "input_audio,input_image,end_image",
        "--height", str(HEIGHT), "--width", str(WIDTH), "--num_frames", str(NUM_FRAMES),
        "--dataset_repeat", str(1 if task == "sft:data_process" else DATASET_REPEAT),
        "--model_id_with_origin_paths", model_paths,
        "--learning_rate", str(LEARNING_RATE),
        "--num_epochs", str(1 if task == "sft:data_process" else NUM_EPOCHS),
        "--remove_prefix_in_ckpt", "pipe.dit.",
        "--output_path", output_path,
        "--lora_base_model", LORA_BASE_MODEL,
        "--lora_target_modules", LORA_TARGET_MODULES,
        "--lora_rank", str(LORA_RANK),
        "--task", task,
    ]
    if dataset_metadata_path:
        cmd += ["--dataset_metadata_path", dataset_metadata_path]
    if USE_GRADIENT_CHECKPOINTING:
        cmd += ["--use_gradient_checkpointing"]
    if USE_GRADIENT_CHECKPOINTING_OFFLOAD:
        cmd += ["--use_gradient_checkpointing_offload"]
    if ENABLE_MODEL_CPU_OFFLOAD:
        cmd += ["--enable_model_cpu_offload"]
    if SILENT_ON_MISSING_AUDIO:
        cmd += ["--silent_on_missing_audio"]
    if SAVE_STEPS:
        cmd += ["--save_steps", str(SAVE_STEPS)]
    if task != "sft:data_process":
        cmd += ["--find_unused_parameters"]
    return cmd

if USE_TWO_STAGE_CACHE:
    stage1_cmd = build_train_command(
        "sft:data_process", DATASET_DIR, os.path.join(DATASET_DIR, "metadata.csv"),
        CACHE_OUTPUT_DIR, MODEL_ID_WITH_ORIGIN_PATHS,
    )
    stage2_cmd = build_train_command(
        "sft:train", CACHE_OUTPUT_DIR, None,
        CHECKPOINT_OUTPUT_DIR, MODEL_ID_WITH_ORIGIN_PATHS,
    )
    train_commands = [("data_process", stage1_cmd), ("train", stage2_cmd)]
else:
    single_cmd = build_train_command(
        "sft", DATASET_DIR, os.path.join(DATASET_DIR, "metadata.csv"),
        CHECKPOINT_OUTPUT_DIR, MODEL_ID_WITH_ORIGIN_PATHS,
    )
    train_commands = [("train", single_cmd)]

for label, cmd in train_commands:
    print(f"--- {label} ---")
    print(" ".join(cmd))


## G — Train

Launches the command(s) built above, streaming logs to
`outputs/experiments/<experiment-name>/logs/`. Does **not** silently
continue after a failure (non-zero exit, including OOM) — it stops and
surfaces the log tail.

In [ ]:
import time

def run_training_stage(label, cmd):
    log_path = os.path.join(LOG_DIR, f"{label}.log")
    print(f"=== starting stage: {label} ===")
    print("log:", log_path)
    t0 = time.time()
    with open(log_path, "w") as log_file:
        process = subprocess.Popen(
            cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        )
        for line in process.stdout:
            log_file.write(line)
            if "it/s]" in line or "loss" in line.lower() or "error" in line.lower():
                print(line.rstrip())
        process.wait()
    elapsed = time.time() - t0
    peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else None
    result = {"label": label, "returncode": process.returncode, "elapsed_sec": round(elapsed, 1), "peak_vram_gb": peak_vram_gb}
    if process.returncode != 0:
        print(f"\n!!! stage '{label}' FAILED (exit {process.returncode}). Last 40 log lines: !!!")
        with open(log_path) as f:
            lines = f.readlines()
        print("".join(lines[-40:]))
        raise RuntimeError(f"Training stage '{label}' failed with exit code {process.returncode}. See {log_path}.")
    print(f"=== stage '{label}' finished in {elapsed:.1f}s, peak VRAM {peak_vram_gb} GB ===")
    return result

# Uncomment to actually launch training. Left inactive by default so this
# notebook can be inspected/edited without accidentally starting a run.
# training_results = [run_training_stage(label, cmd) for label, cmd in train_commands]
print("Training launch is commented out by default. Uncomment the line above when ready to train.")


## H — Validation

Fixed first frame, prompt, seed, resolution, frame count, and inference
settings across checkpoints (per CLAUDE.md validation rules). Generates one
video per selected checkpoint, named by step.

In [ ]:
VALIDATION_SEED = 42  # @param {type:"integer"}
VALIDATION_PROMPT = "pm_motion, character slowly leans toward the camera while maintaining eye contact"  # @param {type:"string"}
VALIDATION_FIRST_FRAME_PATH = "validation/first_frame.png"  # @param {type:"string"}
VALIDATION_INFERENCE_STEPS = 50  # @param {type:"integer"}
VALIDATION_LORA_STRENGTH = 1.0  # @param {type:"number"}

def find_checkpoints(output_dir):
    if not os.path.isdir(output_dir):
        return []
    names = [n for n in os.listdir(output_dir) if n.endswith(".safetensors")]
    return sorted(names)

def run_validation(checkpoint_paths):
    if not os.path.exists(VALIDATION_FIRST_FRAME_PATH):
        raise FileNotFoundError(
            f"Validation first frame not found at {VALIDATION_FIRST_FRAME_PATH}. "
            "Provide validation/first_frame.png before running validation."
        )
    from diffsynth.pipelines.minimax_h3_audio_video import MiniMaxH3Pipeline, ModelConfig
    from diffsynth.utils.data.audio_video import write_video_audio
    from PIL import Image

    vram_config = dict(
        offload_dtype=torch.bfloat16, offload_device="cpu",
        onload_dtype=torch.bfloat16, onload_device="cpu",
        preparing_dtype=torch.bfloat16, preparing_device="cuda",
        computation_dtype=torch.bfloat16, computation_device="cuda",
    )
    pipe = MiniMaxH3Pipeline.from_pretrained(
        torch_dtype=torch.bfloat16, device="cuda",
        model_configs=[
            ModelConfig(model_id="DiffSynth-Studio/MiniMax-H3-NF4", origin_file_pattern=p, **vram_config)
            for p in ["minimax-h3-text-encoder-nf4.safetensors", "minimax-h3-fl2va-nf4.safetensors",
                      "video_vae_nf4.safetensors", "audio_vae_nf4.safetensors"]
        ],
        vram_limit=torch.cuda.mem_get_info("cuda")[1] / (1024 ** 3) - 2,
    )
    first_frame = Image.open(VALIDATION_FIRST_FRAME_PATH)

    produced = []
    for ckpt_name in checkpoint_paths:
        ckpt_path = os.path.join(CHECKPOINT_OUTPUT_DIR, ckpt_name)
        step_label = os.path.splitext(ckpt_name)[0].replace("-", "_")
        out_name = f"{step_label}.mp4"
        out_path = os.path.join(VALIDATION_OUTPUT_DIR, out_name)

        pipe.clear_lora()
        pipe.load_lora(pipe.dit, ckpt_path, alpha=VALIDATION_LORA_STRENGTH)
        video, audio = pipe(
            prompt=VALIDATION_PROMPT,
            height=HEIGHT, width=WIDTH, num_frames=NUM_FRAMES,
            num_inference_steps=VALIDATION_INFERENCE_STEPS, seed=VALIDATION_SEED,
            keyframes=[first_frame], keyframe_indices=[0],
        )
        write_video_audio(video=video, audio=audio, output_path=out_path, fps=FPS, audio_sample_rate=pipe.audio_vae.sample_rate)
        print("saved", out_path)
        produced.append(out_path)
    return produced

# Uncomment once training has produced checkpoints and a real first frame is available.
# checkpoints = find_checkpoints(CHECKPOINT_OUTPUT_DIR)
# validation_outputs = run_validation(checkpoints)
print("Validation is commented out by default. Uncomment once checkpoints exist.")


## I — Experiment summary

In [ ]:
summary = {
    "runtime": {
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "cuda_available": torch.cuda.is_available(),
        "torch_version": torch.__version__,
    },
    "diffsynth_commit": commit_sha,
    "diffsynth_version": diffsynth_version,
    "model_checkpoint": CHECKPOINT_VARIANT,
    "dataset_report": dataset_report,
    "training_settings": config_record,
    "peak_vram_gb": torch.cuda.max_memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else None,
    "checkpoints_produced": find_checkpoints(CHECKPOINT_OUTPUT_DIR),
    "validation_outputs_produced": sorted(os.listdir(VALIDATION_OUTPUT_DIR)) if os.path.isdir(VALIDATION_OUTPUT_DIR) else [],
    "errors_warnings": [],
}
summary_path = os.path.join(EXPERIMENT_DIR, "runtime.json")
with open(summary_path, "w") as f:
    _json.dump(summary, f, indent=2, default=str)
print("Experiment summary written to", summary_path)
_json.dumps(summary, indent=2, default=str)
